### The database as a python package

To import the database there are two options, either by hosting it localy via MongoDB and importing it via database.Database.from_mongodb() or just providing the json file path in database.Database.from_json(filepath). In the first case the default arguments I use are given, but if the url, database name and collection name are different they have to be given to the function. Besides that the backend classes function in the exact same way.

In [ ]:
import pyRMTools as qrm
db = qrm.Database.from_mongodb()

### Inspecting a single AGN

To query a single AGN based on the name the .get(name) command is used. The variable (agn) is then part of the AGN class which has it's measurements as properties and collections. To display them the following syntax is used

In [ ]:
agn = db.get('PG 0052+251')

pos = agn.position
print(['RA', 'DEC'])
print(pos.value)
print(pos.unit)
print()

distance = agn.distance('luminosity distance').measurements
print('D_L')
for dst in distance:
    print(dst.value, dst.unit)
print()

print(r'Inspecting H$\beta$ lags')
lags = agn.lag('H_beta')
print('value', '+', '-', 'Source')
for lag in lags:
    print(lag.value, lag.error_plus, lag.error_minus, qrm.reference_finder(lag.source))
print()

print(r'Inspecting the different H$\beta$ linewidth measurements from rms and mean spectra')
linewidths = agn.linewidth('H_beta', type ='FWHM', spec_type='rms spectrum')
for lw in linewidths:
    print(r'$FWHM_{RMS}$', lw.value, lw.error, lw.unit, qrm.reference_finder(lw.source))
print()

linewidths = agn.linewidth('H_beta', type ='FWHM', spec_type='mean spectrum')
for lw in linewidths:
    print(r'$FWHM_{mean}$',lw.value, lw.error, lw.unit, qrm.reference_finder(lw.source))
print()

linewidths = agn.linewidth('H_beta', type ='line dispersion', spec_type='rms spectrum')
for lw in linewidths:
    print(r'$\sigma_{RMS}$',lw.value, lw.error, lw.unit, qrm.reference_finder(lw.source))
print()

print(r'Inspecting the luminosity measurements at 5100\AA')
luminosities = agn.luminosity(5100)
for lum in luminosities:
    print(lum.value, lum.error, lum.unit, qrm.reference_finder(lum.source))

To all members of the measurement class (e.g. agn.lags('H_beta') is member of the measurement class) we can apply filters with .filter(**kwargs). Multiple filters can be combined by stacking them or giving .filter() multiple arguments. Note that the latter example below is more relevant for AGN with an actual lag grade assigned as Shen2024's 1-5 grading.

In [ ]:
print(r'Inspecting H$\beta$ lags from Woo2024')
lags = agn.lag('H_beta').filter(source = qrm.link_finder('Woo2024'))
print('value', '+', '-', 'Source')
for lag in lags:
    print(lag.value, lag.error_plus, lag.error_minus, qrm.reference_finder(lag.source))
print()

print(r'Inspecting H$\beta$ lags where the lag grade is unknown and Woo2024 measured the lag')
lags = agn.lag('H_beta').filter(grade = 'unknown', source = qrm.link_finder('Woo2024'))
print('value', '+', '-', 'Source')
for lag in lags:
    print(lag.value, lag.error_plus, lag.error_minus, qrm.reference_finder(lag.source))
print()

Masses are a bit of a special measurement, since we not only allow the normal measurement collection as shown in the previous example, but we also allow to combine measurements using the .combine() function. When combining masses they are not a measurement class anymore, but a seperate class. For this class value and error is retrievable aswell as .measurements which links directly to the original measurements. The .measurements class then can be used the same way as masses = agn.mass('H_beta') is used below. And of course filters can be applied beforehand to only combine masses from a specific publication.

Virial products are accessed with the same syntax, but using .vp instead of .mass.

In [ ]:
masses = agn.mass('H_beta')
print('value', '+', '-', 'virial factor', 'spec type')
for m in masses:
    print(m.value, m.error_plus, m.error_minus, m.virial_factor, m.spectrum_type)
print()

print(r'Combined H\beta mass with only rms spectrum meausrements')
mass = agn.mass('H_beta').combine(spectra_type = 'rms')
print('value', '+/-')
print(mass.value, mass.error)
print()

print(r'Combined H\beta mass with only FWHM measurements')
mass = agn.mass('H_beta').combine(linewidth_type = 'FWHM')
print('value', '+/-')
print(mass.value, mass.error)
print()

print(r'Combined H\beta mass with only FWHM measurements from rms spectrum')
mass = agn.mass('H_beta').combine(spectra_type = 'rms', linewidth_type = 'FWHM')
print('value', '+/-')
print(mass.value, mass.error)

### Reviewing publications

The package also offers to view the database from a publications POV. There is multiple things that can be reviewed: 

It is possible to view each AGN the publication has made a measurement for and then iterate through the list which consists of AGN.class and retrieve all the data with the methods shown previously.

It is possible to view each measurement the publication has made, but why would anyone do that?

It is possible to view each measurement the publication has made for a specific AGN.

In [ ]:
'Defining the publication that is viewed'

view = db.publication_view(qrm.link_finder('Grier2017'))

'Retrieve the list of AGNs for which the publication has made measurements and retrieve there names'

agns = view.agns()
names = []
for agn in agns:
    name = agn.name
    names.append(name[0]) # We only print index 0 all names and aliases are stored in a list
print(names)
print()

'Next we can see which measurements Grier2017 has made for the AGN called RM017'

print(view.measurements_by_agn('RM017'))
print()

'For our work it was relevant to compile luminosity and lag measurements from a specific publications'
'which is done as follows:'
l5100 = []
lag_hb = []
name_hb = []

for agn in agns:
    lags = agn.lag('H_beta').filter(source = qrm.link_finder('Grier2017'))
    for lag in lags:
        L = lag.luminosity(5100) # This matching method is described in the next section
        lag_hb.append(lag.value)
        name_hb.append(agn.name[0])
        l5100.append(L.value)

print(name_hb)
print(lag_hb)
print(l5100)

### Matching values with each other

In most cases a retrieved value should be matched with another value. In our case we are interested in matching a lag measurement of a publication with a luminosity measurement of the same publication. This is simply done by accessing the LagMeasurement class and using .luminosity(wavelength).

Alternatively this can be done by using the AGN class and getting the luminosity measurements and then using .match(lag) to get the correct measurement.

Both approaches are generalized, so a measurement collection publication like Bentz2013 will match the lag using the main_reference key, while other publications will use the source key to match the correct value.

In [ ]:
agn = db.get('RM017')

lags = agn.lag('H_beta')#.filter(source = qrm.link_finder('Grier2017'))

print('L5100', r'H\beta lag')
for lag in lags:
    L = lag.luminosity(5100)
    print(L.value, lag.value)
print()
print('L5100', r'H\beta lag')
for lag in lags:
    L = agn.luminosity(5100).match(lag)
    print(L.value, lag.value)

### I have a measurement but I have to know from which AGN it is

In this example we first collect all H$\beta$ lag measurements that exist in the database and store it in a list. Since we don't know which AGN this is for when simply printing the list, we can use the measurement class to get back to the original AGN class by typing .parent. Then we can simply get back to the name by using .name.

In [ ]:
lags = []

for agn in db:

    lag_collection = agn.lag("H_beta")

    if lag_collection:
        lags.extend(lag_collection.measurements)

for lag in lags:
    print(lag.parent.name[0], lag.value)

### Cosmology conversions

If you are unhappy with a cosmology that an author chose it is possible to convert the luminosities using astropy.cosmology. Additionally if you don't want to use the reported (luminosity) distances or in case it is not provided, you can calculate it.

With .cosmology on a luminosity measurement you can also view with which model this has been calculated.

In [ ]:
from astropy.cosmology import LambdaCDM
import astropy.units as u
new_cosmology = LambdaCDM(H0 = 67 * u.km / u.s / u.Mpc, Om0 = 0.32, Ode0 = 0.68)

agn = db.get('PG 0052+251')

luminosities = agn.luminosity(5100)

for lum in luminosities:
    print('Old:', lum.value, lum.cosmology)
    lum.convert(new_cosmology)
    print('New:', lum.value)

print()

print('D_L', agn.distance(cosmology = new_cosmology).value, agn.distance(cosmology = new_cosmology).unit)


### Simulation module

This module contains the relevant ICCF RM simulation. It is used by calling .scout(luminosity, redshift, baseline, cadence, S/N) and it returns the result class. With the result class the various results can be accessed and using results.plot relevant figures can be displayed.

In [ ]:
result = qrm.scout(luminosity = 1e43, z = 0, baseline = 300, cadence = 4, sn = 100)

'To display the recovered lag and errors use'
print('lag', '+', '-')
print(result.lag, result.error_plus, result.error_minus)
print()

'Important statistics as the outlier fraction and success% can be printed by'

print('Outlier fraction:', result.outlier_fraction)
print('Success%:', result.success)
print()

'The bias is returned with'
print('b:', result.bias)
print()

'To view the bias histogram from the simulation use'

result.plot.bias_histogram()

'The visualization of the results on the r-l plane is accessed by'

result.plot.rl_plane()

'A combined view of the bias histogram, the rl-plane and one light curve to review the sampling is possible with'

result.plot.view()

'In each result the ICCF is stored and all ICCF stacked overeach other can be viewed with'

result.plot.iccf()

'To review a single ICCF of the 1000 simulated ones, simply add an index to the function above'

result.plot.iccf(67)
